# Construct quadratic program

In [3]:
import numpy as np
import networkx as nx

from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.primitives import (
    BackendSampler,
    BackendSamplerV2,
    BackendEstimator,
    BackendEstimatorV2,
    StatevectorEstimator,
    Estimator,
)
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import EfficientSU2, TwoLocal
from qiskit_optimization.applications import Maxcut
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import NumPyMinimumEigensolver, VQE
from qiskit_algorithms.optimizers import NELDER_MEAD, COBYLA
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import (
    Sampler as AerSampler,
    SamplerV2 as AerSamplerV2,
    Estimator as AerEstimator,
    EstimatorV2 as AerEstimatorV2,
)

from lcc import LCCVQE

In [4]:
n = 10000
G = nx.random_regular_graph(2, n, seed=0)
max_cut = Maxcut(G)
qp = max_cut.to_quadratic_program()
# print(qp.prettyprint())

## Exact solution

In [20]:
exact = MinimumEigenOptimizer(NumPyMinimumEigensolver())
result_max_cut = exact.solve(qp)
print(result_max_cut.prettyprint())

objective function value: 13.0
variable values: x_0=0.0, x_1=0.0, x_2=1.0, x_3=0.0, x_4=1.0, x_5=1.0, x_6=1.0, x_7=1.0, x_8=0.0, x_9=0.0
status: SUCCESS


# Solve using LCC

In [ ]:
estimator = StatevectorEstimator()
# estimator = None
optimizer = COBYLA()

solver = LCCVQE(
    qp,
    estimator=estimator,
    optimizer=optimizer,
)

initial_params = np.random.rand(2 * qp.get_num_vars()) * 2 * np.pi
print('Initial params:', initial_params.view())
solver.compute_energy(initial_params)

In [16]:
# ansatz = EfficientSU2(
#     num_qubits=qp_max_cut.get_num_vars(),
#     su2_gates='ry',
#     entanglement='circular',
#     reps=1
# )
ansatz = TwoLocal(
    num_qubits=qp.get_num_vars(),
    rotation_blocks='ry',
    entanglement_blocks='cz',
    entanglement='circular',
    reps=1,
)
result = estimator.run([(ansatz, solver.hamiltonian, initial_params)]).result()[0]
result.data.evs

array(0.62592011)

In [ ]:
# estimator = Estimator()
# optimizer = COBYLA()

# solver = LCCVQE(
#     qp_max_cut,
#     estimator=estimator,
#     optimizer=optimizer,
# )

# initial_params = np.random.rand(2 * qp_max_cut.get_num_vars()) * 2 * np.pi
# print('Initial params:', initial_params.view())
# solver.compute_energy_v1(initial_params)

In [ ]:
result = solver.solve(initial_params)
print(result)

In [ ]:
# def obj_func(params):
#     return solver.compute_energy(params)

# result = minimize(obj_func, initial_params, method='cobyla')
# result

In [26]:
print('Expectation:', -(result.fun + solver.offset))

Expectation: 12.999999960222304


In [44]:
solver.offset

-7.5

# Solve without LCC

In [ ]:
ham, offset = qp.to_ising()

backend = AerSimulator(method='statevector')
estimator = Estimator()
ansatz = EfficientSU2(
    num_qubits=qp.get_num_vars(),
    su2_gates='ry',
    entanglement='circular',
    reps=1
)
vqe = VQE(estimator, ansatz, optimizer, initial_point=initial_params)
result = vqe.compute_minimum_eigenvalue(ham)
print(result)

In [111]:
result = estimator.run([(ansatz, ham, initial_params)]).result()[0]
result.data.evs

array(-0.36011752)

# Reconstructing the quasi-prob distribution

## Brute force (run the full circuit)

In [ ]:
backend = AerSimulator(method='matrix_product_state')
sampler = BackendSampler(backend=backend)

num_qubits = qp.get_num_vars()
ansatz = TwoLocal(
    num_qubits=num_qubits,
    rotation_blocks='ry',
    entanglement_blocks='cz',
    entanglement='circular',
    reps=1,
)
ansatz.measure_all()

initial_params = np.random.rand(2 * num_qubits) * 2 * np.pi
result = sampler.run(ansatz, initial_params).result()

In [9]:
backend = AerSimulator(method='matrix_product_state')
sampler = BackendSamplerV2(backend=backend)
# sampler = AerSamplerV2()

num_qubits = qp.get_num_vars()
qc = QuantumCircuit(num_qubits)
thetas = ParameterVector('theta', 2 * num_qubits)
for i in range(num_qubits):
    qc.ry(thetas[i], i)

for i in range(num_qubits - 1):
    qc.cx(i, i + 1)

for i in range(num_qubits):
    qc.ry(thetas[num_qubits + i], i)

qc.measure_all()

initial_params = np.random.rand(2 * num_qubits) * 2 * np.pi
result = sampler.run([(qc, initial_params)]).result()[0]

In [7]:
counts = result.data.meas.get_counts()
max(counts.values())

1

In [1]:
list(map(int, '12345'))

[1, 2, 3, 4, 5]